In [1]:
!pip install dagshub mlflow optuna -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 87.8 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 54.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import mlflow.sklearn
import dagshub
from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from kaggle_secrets import UserSecretsClient
import warnings
warnings.filterwarnings('ignore')

dagshub.init (
    repo_owner="sophyrise",
    repo_name='ieee-cis-fraud-detection',
    mlflow=True
)

mlflow.set_experiment("Gradient Boosting")
print("✅ MLflow tracking URI:", mlflow.get_tracking_uri())

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=2a435cc1-7fda-43d0-9b7c-9303f49b482c&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=1f560b7f7f0f96e806445b978127bc7755dc1e55d8380a8e7fc83fc79baf27f6




Accessing as sophyrise

Initialized MLflow to track repo "sophyrise/ieee-cis-fraud-detection"

Repository sophyrise/ieee-cis-fraud-detection initialized!

2026/05/01 21:11:16 INFO mlflow.tracking.fluent: Experiment with name 'Gradient Boosting' does not exist. Creating a new experiment.


✅ MLflow tracking URI: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow


# **Cleaning**

In [3]:
with mlflow.start_run(run_name="GradientBoosting_Cleaning"):
    import gc
    import numpy as np
    import pandas as pd

    DATA_DIR = "/kaggle/input/competitions/ieee-fraud-detection"
    TXN_MISSING_THRESHOLD = 0.80
    ID_MISSING_THRESHOLD = 0.95
    NEAR_CONSTANT_THRESHOLD = 0.99

    # load
    train_txn = pd.read_csv(f"{DATA_DIR}/train_transaction.csv")
    train_id  = pd.read_csv(f"{DATA_DIR}/train_identity.csv")
    test_txn  = pd.read_csv(f"{DATA_DIR}/test_transaction.csv")
    test_id   = pd.read_csv(f"{DATA_DIR}/test_identity.csv")

    # fix id-01 vs id_01
    test_id.columns = test_id.columns.str.replace("-", "_", regex=False)

    # merge
    train = train_txn.merge(train_id, on="TransactionID", how="left")
    test  = test_txn.merge(test_id, on="TransactionID", how="left")

    del train_txn, train_id, test_txn, test_id
    gc.collect()

    # split target
    y_train = train["isFraud"].astype(np.int8)
    X_train = train.drop(columns=["isFraud", "TransactionID"])
    X_test  = test.drop(columns=["TransactionID"])

    del train, test
    gc.collect()

    # drop high-missing
    id_like_cols = [c for c in X_train.columns if c.startswith("id_") or c in ["DeviceType", "DeviceInfo"]]
    txn_like_cols = [c for c in X_train.columns if c not in id_like_cols]
    missing_ratio = X_train.isnull().mean()

    drop_txn = [c for c in txn_like_cols if missing_ratio[c] > TXN_MISSING_THRESHOLD]
    drop_id  = [c for c in id_like_cols if missing_ratio[c] > ID_MISSING_THRESHOLD]
    drop_missing = drop_txn + drop_id

    X_train = X_train.drop(columns=drop_missing)
    X_test  = X_test.drop(columns=[c for c in drop_missing if c in X_test.columns])

    # drop near-constant
    near_constant_cols = []
    for col in X_train.columns:
        top_freq = X_train[col].value_counts(dropna=False, normalize=True).iloc[0]
        if top_freq > NEAR_CONSTANT_THRESHOLD:
            near_constant_cols.append(col)

    X_train = X_train.drop(columns=near_constant_cols)
    X_test  = X_test.drop(columns=[c for c in near_constant_cols if c in X_test.columns])

    # align test columns
    for col in X_train.columns:
        if col not in X_test.columns:
            X_test[col] = np.nan
    X_test = X_test[X_train.columns]

    # log
    mlflow.log_param("txn_missing_threshold", TXN_MISSING_THRESHOLD)
    mlflow.log_param("id_missing_threshold", ID_MISSING_THRESHOLD)
    mlflow.log_param("near_constant_threshold", NEAR_CONSTANT_THRESHOLD)

    mlflow.log_metric("train_rows", int(X_train.shape[0]))
    mlflow.log_metric("test_rows", int(X_test.shape[0]))
    mlflow.log_metric("final_features", int(X_train.shape[1]))
    mlflow.log_metric("fraud_rate", float(y_train.mean()))
    mlflow.log_metric("dropped_missing_cols", int(len(drop_missing)))
    mlflow.log_metric("dropped_near_constant_cols", int(len(near_constant_cols)))

    print(f"X_train_clean: {X_train.shape}")
    print(f"X_test_clean:  {X_test.shape}")

    # keep for next cells
    X_train_clean = X_train
    X_test_clean = X_test
    y_train_clean = y_train

X_train_clean: (590540, 353)
X_test_clean:  (506691, 353)
🏃 View run GradientBoosting_Cleaning at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/6/runs/2ccb2f924fb74453bf62d7c0f1f08172
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/6


# **Feature Engineering**

In [4]:
with mlflow.start_run(run_name="GradientBoosting_FeatureEngineering"):
    from sklearn.impute import SimpleImputer

    X_train = X_train_clean.copy()
    X_test = X_test_clean.copy()
    y_train = y_train_clean.copy()

    X_train["TransactionAmt_log"] = np.log1p(X_train["TransactionAmt"].clip(lower=0))
    X_test["TransactionAmt_log"] = np.log1p(X_test["TransactionAmt"].clip(lower=0))

    X_train["hour_sin"] = np.sin(2 * np.pi * ((X_train["TransactionDT"] // 3600) % 24) / 24)
    X_train["hour_cos"] = np.cos(2 * np.pi * ((X_train["TransactionDT"] // 3600) % 24) / 24)
    X_test["hour_sin"] = np.sin(2 * np.pi * ((X_test["TransactionDT"] // 3600) % 24) / 24)
    X_test["hour_cos"] = np.cos(2 * np.pi * ((X_test["TransactionDT"] // 3600) % 24) / 24)

    X_train = X_train.drop(columns=["TransactionDT"], errors="ignore")
    X_test = X_test.drop(columns=["TransactionDT"], errors="ignore")

    cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
    num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

    num_imp = SimpleImputer(strategy="median")
    X_train[num_cols] = num_imp.fit_transform(X_train[num_cols])
    X_test[num_cols] = num_imp.transform(X_test[num_cols])

    cat_maps = {}
    for c in cat_cols:
        uniq = pd.Series(X_train[c].astype(str).unique())
        mapping = {v: i for i, v in enumerate(uniq)}
        cat_maps[c] = mapping
        X_train[c] = X_train[c].astype(str).map(mapping).fillna(-1).astype(np.int32)
        X_test[c] = X_test[c].astype(str).map(mapping).fillna(-1).astype(np.int32)

    X_test = X_test.reindex(columns=X_train.columns, fill_value=-1)

    mlflow.log_metric("features_after_fe", int(X_train.shape[1]))
    mlflow.log_metric("cat_cols_encoded", int(len(cat_cols)))

    print("X_train_fe:", X_train.shape)
    print("X_test_fe: ", X_test.shape)

    X_train_fe_gb = X_train
    X_test_fe_gb = X_test
    y_train_fe_gb = y_train

X_train_fe: (590540, 355)
X_test_fe:  (506691, 355)
🏃 View run GradientBoosting_FeatureEngineering at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/6/runs/113e4611da8f45ad945a24eac8bb3f9e
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/6


In [6]:
print(X_train_fe_gb.shape, X_test_fe_gb.shape)
assert X_train_fe_gb.shape[1] == X_test_fe_gb.shape[1]
print("object cols left:", X_train_fe_gb.select_dtypes(include=["object"]).shape[1])

(590540, 355) (506691, 355)
object cols left: 0


# **Feature Selection**

In [7]:
with mlflow.start_run(run_name="GradientBoosting_FeatureSelection"):
    X_train = X_train_fe_gb.copy()
    X_test = X_test_fe_gb.copy()

    X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
    X_test = X_test.replace([np.inf, -np.inf], np.nan).fillna(0)

    nu = X_train.nunique(dropna=False)
    const_cols = nu[nu <= 1].index.tolist()
    X_train = X_train.drop(columns=const_cols, errors="ignore")
    X_test = X_test.drop(columns=const_cols, errors="ignore")

    num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

    sample_n = min(120_000, len(X_train))
    idx = np.random.RandomState(42).choice(len(X_train), size=sample_n, replace=False)
    corr = X_train.iloc[idx][num_cols].corr().abs()

    upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
    drop_corr = [c for c in upper.columns if (upper[c] > 0.98).any()]

    X_train = X_train.drop(columns=drop_corr, errors="ignore")
    X_test = X_test.drop(columns=drop_corr, errors="ignore")

    X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

    mlflow.log_metric("dropped_const", len(const_cols))
    mlflow.log_metric("dropped_corr", len(drop_corr))
    mlflow.log_metric("features_after_fs", int(X_train.shape[1]))

    print("X_train_fs:", X_train.shape)

    X_train_final_gb = X_train
    X_test_final_gb = X_test

X_train_fs: (590540, 306)
🏃 View run GradientBoosting_FeatureSelection at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/6/runs/d44c0960bb3143cf8a1874c0ac14504a
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/6


In [8]:
print(X_train_final_gb.shape, X_test_final_gb.shape)
assert X_train_final_gb.shape[1] == X_test_final_gb.shape[1]

(590540, 306) (506691, 306)


# **Training**

In [9]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import warnings
warnings.filterwarnings('ignore')

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_final_gb, y_train_fe_gb,
    test_size=0.2, random_state=42, stratify=y_train_fe_gb
)
print(f"Train: {X_tr.shape} | Val: {X_val.shape}")

Train: (472432, 306) | Val: (118108, 306)


In [10]:
with mlflow.start_run(run_name="GB_Baseline"):
    mlflow.log_param("n_estimators",  100)
    mlflow.log_param("max_depth",     3)
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_param("subsample",     1.0)
    mlflow.log_param("note",          "default sklearn GradientBoosting params")

    clf = GradientBoostingClassifier(
        n_estimators=100, max_depth=3,
        learning_rate=0.1, subsample=1.0,
        random_state=42
    )
    clf.fit(X_tr, y_tr)

    train_auc = roc_auc_score(y_tr,  clf.predict_proba(X_tr)[:, 1])
    val_auc   = roc_auc_score(y_val, clf.predict_proba(X_val)[:, 1])

    mlflow.log_metric("train_auc",   round(train_auc, 5))
    mlflow.log_metric("val_auc",     round(val_auc,   5))
    mlflow.log_metric("overfit_gap", round(train_auc - val_auc, 5))

    print(f"[Baseline] Train: {train_auc:.4f} | Val: {val_auc:.4f} | Gap: {train_auc - val_auc:.4f}")
    print("  -> sequential boosting, no parallelism — will be slower than XGBoost")

[Baseline] Train: 0.8789 | Val: 0.8758 | Gap: 0.0031
  -> sequential boosting, no parallelism — will be slower than XGBoost
🏃 View run GB_Baseline at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/6/runs/111e91c0021644eaaa97e1e72d5723e3
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/6


In [11]:
# more estimators = more boosting rounds = risk of overfitting without regularization
est_results = []
for n_est in [50, 100, 200]:
    with mlflow.start_run(run_name=f"GB_nestimators_{n_est}"):
        mlflow.log_param("n_estimators",  n_est)
        mlflow.log_param("max_depth",     3)
        mlflow.log_param("learning_rate", 0.1)
        mlflow.log_param("subsample",     0.8)

        clf = GradientBoostingClassifier(
            n_estimators=n_est, max_depth=3,
            learning_rate=0.1, subsample=0.8,
            random_state=42
        )
        clf.fit(X_tr, y_tr)

        train_auc = roc_auc_score(y_tr,  clf.predict_proba(X_tr)[:, 1])
        val_auc   = roc_auc_score(y_val, clf.predict_proba(X_val)[:, 1])
        gap = train_auc - val_auc

        mlflow.log_metric("train_auc",   round(train_auc, 5))
        mlflow.log_metric("val_auc",     round(val_auc,   5))
        mlflow.log_metric("overfit_gap", round(gap, 5))

        est_results.append({"n_estimators": n_est, "train_auc": train_auc,
                             "val_auc": val_auc, "gap": gap})
        print(f"  n_estimators={n_est:<4} | Train: {train_auc:.4f} | Val: {val_auc:.4f} | Gap: {gap:.4f}")

est_df = pd.DataFrame(est_results).set_index("n_estimators")
best_n_est = int(est_df["val_auc"].idxmax())
print(f"\n-> Best n_estimators: {best_n_est}")

  n_estimators=50   | Train: 0.8686 | Val: 0.8658 | Gap: 0.0028
🏃 View run GB_nestimators_50 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/6/runs/0f2c49b0e8a24e60b12f11068c974981
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/6
  n_estimators=100  | Train: 0.8781 | Val: 0.8747 | Gap: 0.0033
🏃 View run GB_nestimators_100 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/6/runs/64e8b4ef4fe34939b7ece5692b78fd76
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/6
🏃 View run GB_nestimators_200 at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/6/runs/74f23331d1934ea4ac15763e457c162f
🧪 View experiment at: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow/#/experiments/6


KeyboardInterrupt: 

In [ ]:
# lower lr = smaller steps = needs more trees but generalizes better
lr_results = []
for lr in [0.3, 0.1, 0.05]:
    with mlflow.start_run(run_name=f"GB_lr_{lr}"):
        mlflow.log_param("n_estimators",  best_n_est)
        mlflow.log_param("max_depth",     3)
        mlflow.log_param("learning_rate", lr)
        mlflow.log_param("subsample",     0.8)

        clf = GradientBoostingClassifier(
            n_estimators=best_n_est, max_depth=3,
            learning_rate=lr, subsample=0.8,
            random_state=42
        )
        clf.fit(X_tr, y_tr)

        train_auc = roc_auc_score(y_tr,  clf.predict_proba(X_tr)[:, 1])
        val_auc   = roc_auc_score(y_val, clf.predict_proba(X_val)[:, 1])
        gap = train_auc - val_auc

        mlflow.log_metric("train_auc",   round(train_auc, 5))
        mlflow.log_metric("val_auc",     round(val_auc,   5))
        mlflow.log_metric("overfit_gap", round(gap, 5))

        lr_results.append({"lr": lr, "train_auc": train_auc, "val_auc": val_auc, "gap": gap})
        print(f"  lr={lr:<5} | Train: {train_auc:.4f} | Val: {val_auc:.4f} | Gap: {gap:.4f}")

lr_df = pd.DataFrame(lr_results).set_index("lr")
best_lr = float(lr_df["val_auc"].idxmax())
print(f"\n-> Best lr: {best_lr}")

In [ ]:
sample_idx = np.random.RandomState(42).choice(len(X_train_final_gb), size=150_000, replace=False)
X_cv = X_train_final_gb.iloc[sample_idx]
y_cv = y_train_fe_gb.iloc[sample_idx]

with mlflow.start_run(run_name="GB_CrossValidation_5fold"):
    mlflow.log_param("n_estimators",  best_n_est)
    mlflow.log_param("max_depth",     3)
    mlflow.log_param("learning_rate", best_lr)
    mlflow.log_param("subsample",     0.8)
    mlflow.log_param("cv_folds",      5)
    mlflow.log_param("cv_sample_size",150_000)

    clf_cv = GradientBoostingClassifier(
        n_estimators=best_n_est, max_depth=3,
        learning_rate=best_lr, subsample=0.8,
        random_state=42
    )

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(clf_cv, X_cv, y_cv, cv=cv, scoring="roc_auc")

    for i, score in enumerate(cv_scores):
        mlflow.log_metric("fold_auc", round(score, 5), step=i)

    mlflow.log_metric("cv_mean_auc", round(cv_scores.mean(), 5))
    mlflow.log_metric("cv_std_auc",  round(cv_scores.std(),  5))

    print(f"CV folds: {[round(s, 4) for s in cv_scores]}")
    print(f"Mean: {cv_scores.mean():.4f} | Std: {cv_scores.std():.4f}")

In [ ]:
with mlflow.start_run(run_name="GB_Final_Pipeline") as run:
    mlflow.log_param("n_estimators",  best_n_est)
    mlflow.log_param("max_depth",     3)
    mlflow.log_param("learning_rate", best_lr)
    mlflow.log_param("subsample",     0.8)
    mlflow.log_param("trained_on",    "full_train_set")

    final_pipe = Pipeline([
        ("clf", GradientBoostingClassifier(
            n_estimators=best_n_est, max_depth=3,
            learning_rate=best_lr, subsample=0.8,
            random_state=42
        ))
    ])
    final_pipe.fit(X_train_final_gb, y_train_fe_gb)

    val_auc = roc_auc_score(y_val, final_pipe.predict_proba(X_val)[:, 1])
    mlflow.log_metric("val_auc", round(val_auc, 5))

    mlflow.sklearn.log_model(
        sk_model=final_pipe,
        artifact_path="gradient_boosting_pipeline",
        registered_model_name="GradientBoosting_FraudDetection"
    )

    print(f"Final Pipeline Val AUC: {val_auc:.4f}")
    print(f"Run ID: {run.info.run_id}")